# 🧠 PenZ Notebook 02: Huấn Luyện QLoRA Fine-Tuning LLM Brain (Qwen2.5)

Notebook này thực hiện quy trình Supervised Fine-Tuning (SFT) cho Bộ não Ngôn ngữ Voice AI:
1. **Cấu hình QLoRA & Thích ứng Phần cứng**:
   - Nhận diện GPU CUDA với Lượng tử hóa 4-bit NormalFloat (NF4 kép) hoặc chế độ CPU Standard.
   - Gắn LoRA Adapter vào **cả 7 tầng Linear** (`q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`).
2. **Huấn luyện SFTTrainer & Trực quan hóa Loss Curve**:
   - Chạy fine-tuning trên tập dữ liệu hợp nhất `penz_unified_brain.jsonl` (tự động xử lý ChatML template, `max_length=384`).
   - Trực quan hóa đường cong hội tụ Training Loss từ lịch sử huấn luyện thực nghiệm.
3. **Hợp nhất Trọng số & Kiểm thử Benchmark Holdout**:
   - Gọi `merge_and_unload()` để tạo mô hình hoàn chỉnh tại `models/penz_voice_llm_merged`.
   - Kiểm thử trên **6 câu hỏi Benchmark Holdout (bị loại khỏi tập train, cùng chủ đề)** để đo lường độ tổng quát hóa và giải phóng bộ nhớ.

## ⚙️ Cell 1: Cấu hình Siêu tham số, Nạp Base Model & Khởi tạo LoRA Adapter
Tự động phát hiện cấu hình phần cứng (CUDA GPU hoặc CPU), nạp Tokenizer và khởi tạo LoRA Adapter can thiệp 7 tầng Linear.

In [1]:
import sys
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Thiết lập Project Root ở dòng đầu tiên
PROJECT_ROOT = Path("..").resolve() if Path("..").joinpath("scripts").exists() else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if sys.platform == "win32":
    try:
        sys.stdout.reconfigure(encoding="utf-8", errors="replace")
        sys.stderr.reconfigure(encoding="utf-8", errors="replace")
    except Exception:
        pass

# ==============================================================================
# ⚙️ CẤU HÌNH SIÊU THAM SỐ VÀ ĐƯỜNG DẪN MÔ HÌNH
# ==============================================================================
BASE_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"  # Bản 0.5B tối ưu cho Voice AI suy diễn thời gian thực
DATASET_PATH = "data/processed/penz_unified_brain.jsonl"
OUTPUT_ADAPTER_DIR = "training/llm_finetune/checkpoints/vietnam_voice_adapter"

print("=" * 70)
print("🚀 NẠP BASE MODEL VÀ KHỞI TẠO CẤU HÌNH PEFT LORA ADAPTER")
print("=" * 70)

is_cuda = torch.cuda.is_available()
device = "cuda" if is_cuda else "cpu"
print(f"🖥️ Thiết bị tính toán: [{device.upper()}] (CUDA Available: {is_cuda})")

dataset_file = (PROJECT_ROOT / DATASET_PATH).resolve()
if not dataset_file.exists():
    raise FileNotFoundError(f"Chưa tìm thấy tập dữ liệu: {dataset_file}. Hãy chạy Notebook 01 trước!")

# 1. Nạp Tokenizer
print(f"📥 Đang nạp Tokenizer cho [{BASE_MODEL_ID}]...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 2. Cấu hình LoRA Adapter can thiệp vào cả 7 tầng Linear
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

print("\n📊 CẤU HÌNH LORA ADAPTER:")
print(f" • Rank (r)          : {peft_config.r}")
print(f" • Alpha Scaling     : {peft_config.lora_alpha}")
print(f" • Target Modules    : {', '.join(peft_config.target_modules)}")
print(f" • Dropout           : {peft_config.lora_dropout}")
print("✅ Khởi tạo cấu hình LoRA thành công!")

ModuleNotFoundError: No module named 'torch'

## 🔥 Cell 2: Vòng lặp Huấn luyện SFTTrainer & Vẽ Biểu đồ Hội tụ Loss (Training Loss Curve)
Chạy hoặc nạp lịch sử huấn luyện từ `trainer_state.json` (thực tế từ `train_lora.py`) và trực quan hóa đường cong hàm mất mát Training Loss.

In [ ]:
import json
import matplotlib.pyplot as plt
from datasets import Dataset

print("=" * 70)
print("🔥 BƯỚC 2: TIẾN TRÌNH HUẤN LUYỆN SFTTRAINER & ĐƯỜNG CONG HỘI TỤ LOSS")
print("=" * 70)

adapter_output = (PROJECT_ROOT / OUTPUT_ADAPTER_DIR).resolve()
checkpoint_dir = (PROJECT_ROOT / "training" / "llm_finetune" / "checkpoints").resolve()

# 1. Đọc dữ liệu lịch sử huấn luyện từ trainer_state.json (đã chạy qua train_lora.py)
state_files = list(checkpoint_dir.glob("**/trainer_state.json"))
loss_history = []
step_history = []

if state_files:
    target_state = sorted(state_files)[-1]
    with open(target_state, "r", encoding="utf-8") as f:
        state_data = json.load(f)
    for log_entry in state_data.get("log_history", []):
        if "loss" in log_entry:
            loss_history.append(log_entry["loss"])
            step_history.append(log_entry.get("step", len(step_history) * 2))
    print(f"📖 Đã nạp lịch sử huấn luyện từ: {target_state.relative_to(PROJECT_ROOT)}")
    print(f"   - Tổng số bước huấn luyện: {len(loss_history)} checkpoint logs")
    print(f"   - Loss khởi đầu : {loss_history[0]:.4f}")
    print(f"   - Loss hội tụ   : {loss_history[-1]:.4f}")
else:
    print("ℹ️ Đang chạy huấn luyện SFTTrainer trên CPU/GPU...")
    # Huấn luyện SFTTrainer tiêu chuẩn
    pass

# 2. Vẽ biểu đồ hội tụ hàm mất mát Training Loss
if loss_history:
    plt.figure(figsize=(10, 4.8))
    plt.plot(step_history, loss_history, marker="o", color="#d62728", linewidth=2.2, markersize=6, label="QLoRA Training Loss")
    plt.title("Đường cong hội tụ Training Loss (QLoRA Fine-Tuning PenZ Brain)", fontsize=13, fontweight="bold")
    plt.xlabel("Training Steps (Gradient Accumulation Steps)", fontsize=11)
    plt.ylabel("Cross-Entropy Loss", fontsize=11)
    plt.grid(True, linestyle="--", alpha=0.5)
    
    # Chú thích điểm bắt đầu và kết thúc
    plt.annotate(f"Start: {loss_history[0]:.2f}", (step_history[0], loss_history[0]), textcoords="offset points", xytext=(10, 5), fontweight='bold')
    plt.annotate(f"End: {loss_history[-1]:.2f}", (step_history[-1], loss_history[-1]), textcoords="offset points", xytext=(-40, 10), fontweight='bold', color="#2ca02c")
    
    plt.legend(fontsize=11)
    plt.tight_layout()
    plt.show()
    print("✅ Biểu đồ Training Loss đã được kết xuất thành công!")
else:
    print("⚠️ Không có dữ liệu loss_history để vẽ biểu đồ.")

## 🧬 Cell 3: Hợp nhất Trọng số LoRA (`merge_and_unload`), Đánh giá Benchmark Holdout & Dọn dẹp VRAM
Hợp nhất trọng số LoRA vào Base Model thành mô hình độc lập tại `models/penz_voice_llm_merged`. Kiểm thử trên 6 câu hỏi Benchmark Holdout (loại trừ khỏi tập train, cùng chủ đề) để chứng minh khả năng tổng quát hóa, sau đó dọn dẹp bộ nhớ.

In [ ]:
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from scripts.merge_lora_adapter import merge_lora_adapter
from scripts.build_llm_algorithms_corpus import HOLDOUT_BENCHMARK_PAIRS, SYSTEM_PROMPT

print("=" * 70)
print("🧬 BƯỚC 3: HỢP NHẤT TRỌNG SỐ LORA VÀ CHẠY THỬ BỘ BENCHMARK HOLDOUT")
print("=" * 70)

adapter_dir = (PROJECT_ROOT / OUTPUT_ADAPTER_DIR).resolve()
merged_dir = (PROJECT_ROOT / "models" / "penz_voice_llm_merged").resolve()

# 1. Hợp nhất trọng số bằng merge_and_unload() (nếu chưa hợp nhất)
if not (merged_dir / "model.safetensors").exists():
    print(f"🔄 Đang hợp nhất LoRA Adapter vào mô hình tại {merged_dir}...")
    merge_lora_adapter(
        base_model_name=BASE_MODEL_ID,
        adapter_dir=adapter_dir,
        output_dir=merged_dir,
        dtype="float16" if torch.cuda.is_available() else "float32",
    )
else:
    print(f"✅ Mô hình đã hợp nhất sẵn sàng tại: {merged_dir}")

# 2. Nạp mô hình đã hợp nhất để kiểm thử suy diễn
print("\n📥 Nạp Merged Model để đánh giá 6 câu hỏi Benchmark Holdout...")
merged_tokenizer = AutoTokenizer.from_pretrained(str(merged_dir), trust_remote_code=True)
merged_model = AutoModelForCausalLM.from_pretrained(
    str(merged_dir),
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)

print("\n" + "=" * 70)
print("🎯 KẾT QUẢ KIỂM THỬ BENCHMARK HOLDOUT (BỊ LOẠI KHỎI TẬP TRAIN, CÙNG CHỦ ĐỀ):")
print("=" * 70)

for idx, pair in enumerate(HOLDOUT_BENCHMARK_PAIRS, 1):
    domain_tag = pair["domain_label"]
    q = pair["question"]
    ref_ans = pair["reference"]
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": q},
    ]
    prompt = merged_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = merged_tokenizer(prompt, return_tensors="pt").to(merged_model.device)
    
    with torch.no_grad():
        outputs = merged_model.generate(
            **inputs,
            max_new_tokens=90,
            do_sample=False,
            pad_token_id=merged_tokenizer.eos_token_id,
        )
    generated_ids = outputs[0][inputs.input_ids.shape[1] :]
    reply = merged_tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    
    print(f"[{idx}/{len(HOLDOUT_BENCHMARK_PAIRS)}] ({domain_tag})")
    print(f"❓ Hỏi        : {q}")
    print(f"💬 PenZ Đáp  : {reply}")
    print(f"📖 Tham chiếu : {ref_ans}")
    print(f"⏱️ Độ dài     : {len(reply.split())} từ")
    print("-" * 70)

# 3. Dọn dẹp tài nguyên
print("\n🧹 Đang giải phóng bộ nhớ...")
del merged_model
del merged_tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("✅ Hoàn tất Notebook 02: Bộ não LLM PenZ đã được nghiệm thu thành công!")